# 2.2 Comparing categories

**The question: how do these groups differ?**

Half of this notebook is the toolkit — bar charts, grouped bars, heatmaps, and the design
decisions that make them readable. The other half is two of the ways a group comparison
misleads even when every number in it is correct. A third way — the one that needs its own
notebook — is next: [02.3-simpsons_paradox](02.3-simpsons_paradox.ipynb).

In [ ]:
import pandas as pd
import seaborn as sns
from goad_toolkit.visualizer import (
    BasePlot,
    GroupedBarPlot,
    HeatmapPlot,
    HighlightCategory,
    PlotSettings,
)

from wa_analyzer.data import load_showcase

## 2.2.1 A `BarPlot` worth keeping

[02.1-baseplot-101](02.1-baseplot-101.ipynb) showed the shape of a `BasePlot` subclass on a
throwaway dataset. Same five lines, once more — this time it is the one the rest of this
notebook actually uses.

In [ ]:
class BarPlot(BasePlot):
    """A bar chart. All the styling lives in PlotSettings."""

    def build(self, data: pd.DataFrame, x: str, y: str, **kwargs):
        sns.barplot(data=data, x=x, y=y, ax=self.ax, **kwargs)
        return self.fig, self.ax

## 2.2.2 The showcase: penguins

Three species, two sexes, three islands, 344 birds. Small enough to check by hand and
different enough that a bad chart cannot hide the differences.

In [ ]:
penguins = load_showcase("penguins")
penguins.head()

### Why look at the head before plotting anything

Not out of habit — to answer three questions a plot call will silently get wrong if you
don't: What is a row here, one bird? What type is each column — is `species` really three
fixed levels, is `body_mass_g` really a number and not a string with units stuck to it? And
is anything obviously missing — a `NaN` in `sex`, an island that looks misspelled? `.head()`
is the cheapest place to catch a column that means something other than its name suggests,
before you have built a chart around the wrong assumption.

Here: three species, three islands, two sexes, and four continuous measurements in
millimetres and grams. That is what makes `species` the natural first thing to group by
below.

In [ ]:
settings = PlotSettings(
    figsize=(8, 4),
    title="Body mass by species",
    xlabel="species",
    ylabel="body mass (g)",
)
fig, ax = BarPlot(settings).plot(data=penguins, x="species", y="body_mass_g")

> **Try it.** Change `xlabel` and `ylabel` above — to `"penguin species"` and
> `"average body mass (grams)"`, say — and rerun the cell. Nothing else has to change; this is
> the whole point of keeping the makeup in `PlotSettings`. Pick labels a reader would actually
> want, not just the column names back.

### Order is a decision

Seaborn plotted the species in the order it found them. Alphabetical order, or
order-of-appearance, is almost never the order your reader needs.

**Sort by the value unless the category has its own sequence** — days of the week, age
bands, months. An unsorted bar chart makes the reader do the comparison you were supposed to
do for them.

In [ ]:
order = penguins.groupby("species").body_mass_g.mean().sort_values(ascending=False).index

fig, ax = BarPlot(settings).plot(data=penguins, x="species", y="body_mass_g", order=order)

### Grey first, then one colour

The fifth of the course's five guidelines. Draw everything in grey, then colour the one
thing you are talking about. Colour used this way is a pointer; colour used everywhere is
decoration that costs the reader attention and returns nothing.

`HighlightCategory` is that idea as a layer, from 02.1: it recolours the bars already on the
axis, so it works over your `BarPlot`, over a bare `sns.barplot`, over anything that drew
bars. Which category to pick out is `PlotSettings(highlight=[...])`, alongside the rest of
the makeup.

Three statements rather than one chained expression, deliberately: build the settings, build
the plot, then layer. Each line does one thing and each is readable on its own.

In [ ]:
highlight = "Gentoo"

settings = PlotSettings(
    figsize=(8, 4),
    title=f"{highlight} are the heavy ones",
    xlabel="species",
    ylabel="body mass (g)",
    highlight=[highlight],
)

bars = BarPlot(settings)
bars.plot(data=penguins, x="species", y="body_mass_g", order=order)
bars.plot_on(HighlightCategory(settings))

The `BarPlot` above is not a novelty: `goad_toolkit` ships an identical one, along with
`GroupedBarPlot`, `HeatmapPlot` and `BarbellPlot`. Writing it here is the exercise, the same
move as `RegexFeature` in lesson 1 — a `BasePlot` subclass is five lines, and having written
those five lines once, the shipped classes stop being a black box you called and start being
something you can read, argue with the defaults of, and extend when nothing fits.

From lesson 3 on, this notebook imports them instead of rederiving them.

### Two categorical variables at once

A grouped bar when both have few levels; a heatmap when either has many. Both ship, so
neither needs a `build` of your own — `GroupedBarPlot` and `HeatmapPlot`.

They also go side by side, and that is `BasePlot`'s other half from 02.1.
`create_figure(n_plots=2)` makes the grid and hands back the axes; `plot_on_axes` puts a plot
on a named one. No `plt.subplots`, and the panel titles come from `settings.subplot_titles`.

In [ ]:
pair = PlotSettings(
    figsize=(13, 4),
    title="Three species, two other variables",
    subplot_titles=["grouped bars: 3 species x 2 sexes", "heatmap: who lives where"],
    xlabel="species",
    ylabel="body mass (g)",
)

host = BarPlot(pair)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(GroupedBarPlot(pair), axes[0],
                  data=penguins, x="species", y="body_mass_g", hue="sex", order=order)
host.plot_on_axes(HeatmapPlot(pair), axes[1],
                  data=penguins, index="species", columns="island",
                  values="body_mass_g", aggfunc="size", fmt=".0f", cmap="Blues")
fig.tight_layout()

The heatmap answers a question the bars cannot: Gentoo and Chinstrap each live on exactly
one island, Adelie on all three. That is a fact about the *design of the data*, and it is
the kind of thing worth knowing before you compare anything — a species/island comparison is
partly a comparison of islands.

### This class moves to `scripts/plots.py`

Same reasoning as `ParseIRCLines` in lesson 1: a class defined only in a notebook cell cannot
be imported anywhere else, and a dashboard is exactly the kind of "anywhere else" that needs
it. `BarPlot` above is identical to what ships in `goad_toolkit.visualizer` — the derivation
was the exercise — so it moves to `scripts/plots.py` unchanged. Lesson 7's dashboard imports
it from there and renders it inside streamlit without touching a line of it, because `plot()`
returns an ordinary matplotlib figure and `st.pyplot()` takes one as-is.

In [ ]:
from scripts.plots import BarPlot as ImportedBarPlot

# The same class, from the file rather than from the cell above — the figure is identical.
fig, ax = ImportedBarPlot(settings).plot(
    data=penguins, x="species", y="body_mass_g", order=order, color="#cccccc"
)

## 2.2.3 Two ways a group comparison misleads

Same numbers, different chart, different (wrong) impression. The showcase for this half is
the Titanic passenger list — one row per passenger, whether they survived, and which class,
deck and port they boarded from.

In [ ]:
titanic = load_showcase("titanic")
titanic.head()

891 passengers, 15 columns; `deck` and `age` both have real gaps, which matters for what
follows. Two questions, one dataset:

- Which combination of deck and class carried the most passengers? — an ordinary counting
  question, and about to go wrong for a reason that has nothing to do with counting.
- Which deck had the best survival rate? — a proportion, and about to go wrong for the
  oldest reason a proportion goes wrong.

### Too many bars, in no order

In [ ]:
# .size() on a groupby always returns a Series, whose reset_index(name=...) is valid;
# ty infers DataFrame here and picks the wrong overload.
by_deck = titanic.assign(deck=titanic.deck.astype(object).fillna("unknown"))
by_deck = by_deck.groupby(["deck", "class"], observed=True).size().reset_index(name="n")  # ty: ignore[no-matching-overload]

decks = PlotSettings(
    figsize=(13, 4),
    title="Same passengers, two ways of asking",
    subplot_titles=["unordered, rainbow, 14 bars", "one question, ordered, one colour"],
    xlabel="passengers",
    ylabel="",
)

host = BarPlot(decks)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(GroupedBarPlot(decks), axes[0],
                  data=by_deck, x="n", y="deck", hue="class", palette="rainbow")

top = titanic.groupby("class", observed=True).size().sort_values(ascending=False)  # ty: ignore[no-matching-overload]
host.plot_on_axes(BarPlot(decks), axes[1],
                  data=top.reset_index(name="n"), x="n", y="class", color="#cccccc")
fig.tight_layout()

Notice the deck labelled **`unknown`**. A bare `groupby(["deck", "class"])` would silently
drop every row where `deck` is missing, and for third class that is 479 of 491 passengers:
cabins were almost never logged down in steerage, so third class barely exists in a
deck-based chart unless you say so. That is also the answer to "where did the rest of third
class go" in the right panel — it was never in the left one to begin with, only missing from
it. Filling the gap in as its own value, rather than dropping it, is what "show the data"
actually requires: the alternative is a chart that quietly reports on two-thirds of the ship
and calls it a comparison.

The left chart answers "which deck-and-class combination carried the most passengers" with
21 bars in 21 colours and no order — technically an answer, practically unreadable. The right
chart answers a narrower, sharper question — which *class* carried the most passengers — and
you can read it in one glance. The left chart is not wrong, but it is just harder to communicate a point.

### A proportion without its denominator

Now the second question: which deck had the best survival rate. Watch what happens when a
rate gets plotted without the count it rests on.

In [ ]:
rate = titanic.groupby("deck", observed=True).survived.agg(["mean", "size"])
rate = rate.sort_values("mean", ascending=False).reset_index()
rate["pct"] = rate["mean"] * 100

proportion = PlotSettings(
    figsize=(13, 4),
    title="A rate, and what it rests on",
    subplot_titles=["survival rate by deck (%)", "...how many people that is"],
    xlabel="",
    ylabel="deck",
)

host = BarPlot(proportion)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(BarPlot(proportion), axes[0],
                  data=rate, x="pct", y="deck", color="#cccccc")
host.plot_on_axes(BarPlot(proportion), axes[1],
                  data=rate, x="size", y="deck", color="#cccccc")
fig.tight_layout()

Deck G has a survival rate you could write a headline about, and four people on it.

Whenever you plot a proportion, plot or annotate the count behind it. A rate over a handful
of cases is mostly noise, and the bar chart gives it exactly the same visual weight as a
rate over hundreds — which is exactly what the left panel just did, and exactly what the
right panel exists to catch.

**Ask this every time you plot a rate:** how many observations is this percentage actually
standing on?

---

**Where this goes next.** These two failures share one shape: a summary that hides the thing
that would have changed your mind about it. There is a third, and it does not need a
carelessly chosen chart to appear — it happens when the numbers themselves reverse depending
on how you group, and every step along the way is correct. That one gets a notebook of its
own: [02.3-simpsons_paradox](02.3-simpsons_paradox.ipynb).